In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FraudDetection_Comparison") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

#Load data
identity_path    = "../data/raw/train_identity.csv"
transaction_path = "../data/raw/train_transaction.csv"

identity_df    = spark.read.csv(identity_path,    header=True, inferSchema=True)
transaction_df = spark.read.csv(transaction_path, header=True, inferSchema=True)

print("=" * 60)
print(f"Identity    : {identity_df.count():>8,} rows | {len(identity_df.columns)} cols")
print(f"Transaction : {transaction_df.count():>8,} rows | {len(transaction_df.columns)} cols")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 22:13:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Identity    :  144,233 rows | 41 cols
Transaction :  590,540 rows | 394 cols


In [2]:
from pyspark.sql import functions as F

#2. Drop null columns (>90% null)
def drop_null_columns(df, threshold=0.9, label=""):
    total_rows = df.count()
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    cols_to_drop = [c for c, cnt in null_counts.items()
                    if (cnt / total_rows) > threshold]
    df = df.drop(*cols_to_drop)
    print(f"[{label}] Dropped {len(cols_to_drop)} cols → còn {len(df.columns)} cols")
    return df

print("\n--- Drop null columns ---")
transaction_df = drop_null_columns(transaction_df, label="TRANSACTION")
identity_df    = drop_null_columns(identity_df,    label="IDENTITY")



--- Drop null columns ---


26/04/21 22:13:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
ERROR:root:KeyboardInterrupt while sending command.               (0 + 16) / 16]
Traceback (most recent call last):
  File "/home/tmkhoa1812/miniconda3/envs/base_python_3.10/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/tmkhoa1812/miniconda3/envs/base_python_3.10/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/tmkhoa1812/miniconda3/envs/base_python_3.10/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
#3. Inner join trên TransactionID
joined_df = transaction_df.join(identity_df, on="TransactionID", how="inner")
print(f"\nJoined : {joined_df.count():,} rows | {len(joined_df.columns)} cols")

In [ ]:
from pyspark.sql.types import (IntegerType, DoubleType, FloatType,
                                LongType, ShortType, ByteType,
                                DecimalType, StringType)

#4. Fill null
#Numeric columns -> fill mean/-1
#Text -> fill UNKNOWN

SKIP = {"TransactionID", "isFraud"}
numerical_types = (IntegerType, DoubleType, FloatType, LongType,
                   ShortType, ByteType, DecimalType)

num_cols = [f.name for f in joined_df.schema.fields
            if isinstance(f.dataType, numerical_types) and f.name not in SKIP]
str_cols = [f.name for f in joined_df.schema.fields
            if isinstance(f.dataType, StringType)      and f.name not in SKIP]

mean_row = joined_df.agg(*[F.mean(c).alias(c) for c in num_cols]).collect()[0].asDict()

fill_mean   = {c: v    for c, v in mean_row.items() if v is not None}
fill_minus1 = {c: -1.0 for c in mean_row            if mean_row[c] is None}

joined_df = (joined_df
             .fillna(fill_mean)
             .fillna(fill_minus1)
             .fillna({c: "UNKNOWN" for c in str_cols}))

print(f"\n--- Fill nulls ---")
print(f"  Num cols filled với mean : {len(fill_mean)}")
print(f"  Num cols filled với -1   : {len(fill_minus1)}")
print(f"  Str cols filled UNKNOWN  : {len(str_cols)}")

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from tqdm.auto import tqdm

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in str_cols
]

encoded_df = joined_df
print("\nEncoding categorical features:")
for indexer in tqdm(indexers, desc="Applying StringIndexers"):
    encoded_df = indexer.fit(encoded_df).transform(encoded_df)

encoded_df = encoded_df.drop(*str_cols)
for c in str_cols:
    encoded_df = encoded_df.withColumnRenamed(c + "_idx", c)

print("\nEncoding hoàn tất.")

In [ ]:
from pyspark.sql.functions import col as F_col

feature_cols = [c for c in encoded_df.columns if c not in SKIP]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

ml_df = (assembler
         .transform(encoded_df)
         .select("features", F_col("isFraud").cast("double").alias("label")))

# Train/Test split 80/20
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
print(f"\nTrain : {train_df.count():,} | Test : {test_df.count():,}")


In [ ]:
import pandas as pd

def save_feature_schema_to_csv(ml_df, original_df, file_name="feature_schema.csv"):
    # 1. Lấy metadata từ cột features của mô hình
    meta = ml_df.schema["features"].metadata["ml_attr"]["attrs"]

    feature_data = []
    for attr_type in meta:
        for attr in meta[attr_type]:
            name = attr["name"]
            # Data type
            dtype = original_df.schema[name].dataType.simpleString()
            feature_data.append({"feature_name": name, "data_type": dtype})

    # 2. Tạo DataFrame và sắp xếp
    schema_df = pd.DataFrame(feature_data).sort_values("feature_name")

    # 3. Lưu ra CSV
    schema_df.to_csv(file_name, index=False)
    print(f"Đã lưu schema vào file: {file_name}")
    return schema_df

# Thực thi và hiển thị 5 dòng đầu
df_schema = save_feature_schema_to_csv(ml_df, joined_df)
print(df_schema.head())

In [ ]:
total    = train_df.count()
n_fraud  = train_df.filter(F_col("label") == 1).count()
n_normal = total - n_fraud
weight   = n_normal / n_fraud

print(f"\n--- Class imbalance ---")
print(f"  Normal : {n_normal:,}  ({n_normal/total*100:.1f}%)")
print(f"  Fraud  : {n_fraud:,}   ({n_fraud/total*100:.1f}%)")
print(f"  Weight cho class 1 : {weight:.1f}x")

train_weighted = train_df.withColumn(
    "classWeight",
    F.when(F_col("label") == 1, float(weight)).otherwise(1.0)
)

In [ ]:
import matplotlib.pyplot as plt
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

def run_gbt_overfitting_analysis(train_df, test_df):
    print("\n--- Đang chạy GBT với phân tích Overfitting ---")

    # Các mốc dữ liệu để kiểm tra (20%, 40%, 60%, 80%, 100%)
    fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
    train_scores = []
    test_scores = []

    # Sử dụng PR-AUC vì dữ liệu gian lận bị mất cân bằng (imbalanced)
    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")

    for frac in fractions:
        # Lấy mẫu dữ liệu
        sample_train = train_df.sample(withReplacement=False, fraction=frac, seed=42)

        # Cấu hình GBT: Giảm maxDepth và maxIter để tiết kiệm dung lượng
        gbt = GBTClassifier(
            featuresCol="features",
            labelCol="label",
            weightCol="classWeight",
            maxIter=20,     # Số lượng cây (giảm xuống để tránh tốn RAM)
            maxDepth=5,     # Độ sâu thấp giúp kiểm soát overfitting và bộ nhớ
            stepSize=0.1,   # Learning rate
            maxBins=2000,   # Tăng maxBins để xử lý các biến phân loại có nhiều giá trị
            seed=42
        )

        model = gbt.fit(sample_train)

        # Dự đoán
        train_preds = model.transform(sample_train)
        test_preds = model.transform(test_df)

        # Tính điểm PR-AUC
        train_auc = evaluator.evaluate(train_preds)
        test_auc = evaluator.evaluate(test_preds)

        train_scores.append(train_auc)
        test_scores.append(test_auc)
        print(f"Dữ liệu {int(frac*100):>3}%: Train PR-AUC = {train_auc:.4f} | Test PR-AUC = {test_auc:.4f}")

    # Vẽ biểu đồ Learning Curve
    plt.figure(figsize=(10, 6))
    plt.plot(fractions, train_scores, 's--', color="blue", label="Training PR-AUC")
    plt.plot(fractions, test_scores, 'o-', color="orange", label="Test PR-AUC")
    plt.title("GBT Learning Curve: Kiểm tra Overfitting")
    plt.xlabel("Tỷ lệ dữ liệu huấn luyện")
    plt.ylabel("PR-AUC Score")
    plt.legend(loc="lower right")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()

    return model

# Thực thi
gbt_final_model = run_gbt_overfitting_analysis(train_weighted, test_df)

In [ ]:
# Đường dẫn thư mục bạn muốn lưu model trên Drive
model_path = "../../../model_ml/model/gbt_model"

# Lưu mô hình
# .overwrite() giúp ghi đè nếu thư mục đã tồn tại, tránh lỗi
gbt_final_model.write().overwrite().save(model_path)
#
print(f"Mô hình đã được lưu thành công tại: {model_path}")

In [ ]:
from pyspark.ml.classification import GBTClassificationModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print(f"--- Đang kiểm tra mô hình tại: {model_path} ---")

try:
    # 1. Gọi model đã lưu lên
    loaded_model = GBTClassificationModel.load(model_path)
    print("1. Nạp mô hình thành công!")

    # 2. Chạy dự đoán thử trên tập test_df
    predictions = loaded_model.transform(test_df)

    # 3. Hiển thị kết quả dự đoán (xem các cột probability và prediction)
    print("2. Kết quả dự đoán mẫu:")
    predictions.select("label", "prediction", "probability").show(5)

    # 4. Kiểm tra lại hiệu năng (PR-AUC) để đảm bảo không sai lệch
    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
    test_auc = evaluator.evaluate(predictions)
    print(f"3. Kiểm tra PR-AUC sau khi load: {test_auc:.4f}")

except Exception as e:
    print(f"Lỗi khi kiểm tra mô hình: {e}")